## Conservation

### Data Preparation

In [ ]:
# import dask.dataframe as dd
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import csv
import math
from pathlib import Path
import itertools
from matplotlib.ticker import MaxNLocator
from tqdm import tqdm
from collections import Counter, defaultdict
import seaborn as sns 
import pandas 
import os 
import math
import os
import pandas as pd
import gzip


In [ ]:
import matplotlib
from pathlib import Path
import os

indir = Path(f"{os.getenv('SCRATCH')}/g4_t2t_revisions_data")
target = Path(os.getenv("WORK")).joinpath("g4_revisions/g4_t2t_revisions/G4_T2T/figures/haplotypes")
target.mkdir(exist_ok=True, parents=True)
dataset_path = Path(os.getenv("SCRATCH")) / "g4_t2t_revisions_data"
G4HUNTER = dataset_path / "pG4s_extractions" / "g4hunter" / "chm13v2_g4hunter.txt.gz"
REGEX = dataset_path / "pG4s_extractions" / "quadparser" / "chm13v2_regex_motifs.txt"

g4_df = pd.read_table(G4HUNTER)
regex_df = pd.read_table(REGEX)
g4_df

In [ ]:
haplotype_colors = {"paternal": "#d687d1",
                     "maternal": "#f5eba2"
                     }
sex_colors = {
               "XY": "#455aba",
              "XX": "#92d4b7"
             }

superpopulation_colors = {
              "EAS": "#ed1a76",
              "AMR": "#fcba03",
              "SAS": "#036bfc",
              "AFR": "#169e87"
             }

In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

haplotype_colors = {"paternal": "#d687d1",
                     "maternal": "#f5eba2"
                     }
sex_colors = {
               "XY": "#455aba",
              "XX": "#92d4b7"
             }

superpopulation_colors = {
              "EAS": "#ed1a76",
              "AMR": "#fcba03",
              "SAS": "#036bfc",
              "AFR": "#169e87"
             }
fig_dir = Path(os.getenv("SCRATCH")) / "figures_g4_t2t"
fig_dir.mkdir(exist_ok=True)

legend_configs = {
    "legend_haplotype": haplotype_colors,
    "legend_sex": sex_colors,
    "legend_superpopulation": superpopulation_colors,
}

for fname, color_dict in legend_configs.items():
    handles = [mpatches.Patch(facecolor=c, label=l) for l, c in color_dict.items()]
    fig, ax = plt.subplots(figsize=(2, len(color_dict) * 0.5))
    ax.axis("off")
    ax.legend(
        handles=handles,
        loc="center",
        fontsize=16,
        frameon=True,
        fancybox=True,
        handlelength=1.5,
        handleheight=1.5,
    )
    fig.savefig(target/ f"{fname}.pdf", bbox_inches="tight", transparent=True)
    plt.show()


In [ ]:
import polars as pl
conservation_df = pl.read_csv(
   f"{os.getenv('SCRATCH')}/MAFin_results_CHM13_g4/CHM13_pG4s_CHM13.g4hunter.unique_pG4s_CHM13.g4hunter.unique_motif_hits.csv.gz"
)

def complement(seq):
    seq = seq.upper()
    complement_seq = seq.translate(str.maketrans("ACGT", "TGCA"))[::-1]
    return min(seq, complement_seq)
conservation_df = conservation_df.with_columns(
                    pl.col("motif_name")
                .map_elements(lambda seq:
                                        complement(seq.upper()),  return_dtype=pl.Utf8)
                .alias("canonical")
)
conservation_df

In [ ]:
haplotypes = pd.read_table("https://raw.githubusercontent.com/human-pangenomics/HPP_Year1_Data_Freeze_v1.0/refs/heads/main/sample_metadata/hprc_year1_sample_metadata.txt")
ancestries = dict(zip(haplotypes["Sample"], 
                      haplotypes["Superpopulation"]))
len(ancestries)
haplotypes

In [ ]:
haplotypes.loc[haplotypes["Sample"] == "NA21309", "Superpopulation"] = "AFR"
ancestries = dict(zip(haplotypes["Sample"], haplotypes["Superpopulation"]))
valid_haplotypes = [col for col in conservation_df.columns if col.split('#')[0] in ancestries]

sex_pop = haplotypes.set_index("Sample")["Sex"]
sex_pop.value_counts()

sex_pop_meta = sex_pop.to_dict()
sex_pop_meta

In [ ]:
from pathlib import Path 

infiles = [infile for infile in Path(os.getenv("SCRATCH")).joinpath("G4Hunter-Companion/G4_new_results").glob("*_pG4s.g4_hunter.tsv")]
infiles_quad = [infile for infile in Path(os.getenv("SCRATCH")).joinpath("G4Hunter-Companion/G4_new_results").glob("*_pG4s.consensus.tsv")]
processed_infiles = dict()
processed_infiles_quad = dict()

# # #
for infile in infiles:
    haplotype = infile.stem.split(".")[0]
    origin = infile.stem.split(".")[1]
    if haplotype + "#1" in valid_haplotypes or haplotype + "#2" in valid_haplotypes:
        processed_infiles[haplotype + "#" + origin] = (infile, haplotype, origin)
    else:
        print(haplotype)

for infile in infiles_quad:
    haplotype = infile.stem.split(".")[0]
    origin = infile.stem.split(".")[1]
    if haplotype + "#1" in valid_haplotypes or haplotype + "#2" in valid_haplotypes:
        processed_infiles_quad[haplotype + "#" + origin] = (infile, haplotype, origin)
    else:
        print(haplotype)
len(processed_infiles_quad)

In [ ]:
haplotypes_fasta = [file for file in Path(os.getenv("SCRATCH")).joinpath("year1_pangenome").glob("*.fa.gz") if file.name.split(".")[0] + "#1" in valid_haplotypes]
len(haplotypes_fasta)

In [ ]:
# genome_sizes = defaultdict(int)
# nucleotides = {"a", "g", "c", "t", "A", "G", "C", "T"}
# for haplotype in tqdm(haplotypes_fasta):
#     pop = haplotype.name.split(".")[0]
#     origin = haplotype.name.split(".")[1]
#     haplotype_id = pop + "#" + origin
#     for seqID, seq in tqdm(parse_fasta(haplotype), total=25, leave=True):
#         seq = seq.upper()
#         genome_sizes[haplotype_id] += sum(seq.count(c) for c in "ACGT")
# genome_sizes_df = pd.DataFrame.from_dict(genome_sizes, 
#                                          orient="index", 
#                                          columns=["genome_size"])
# genome_sizes_df
# genome_sizes_df.to_csv(f"{target}/genome_sizes_hprc_haplotypes.csv.gz", 
#                        compression="gzip", 
#                        sep=",", 
#                        mode="w")

In [ ]:
f"{target}/g4_revisions/g4_t2t_revisions/conservation_analysis/genome_sizes_hprc_haplotypes.csv"

In [ ]:
target = Path(os.getenv("WORK"))
genome_sizes_df_ = pd.read_csv(f"{target}/g4_revisions/g4_t2t_revisions/conservation_analysis/genome_sizes_hprc_haplotypes.csv", index_col=0)
genome_sizes_df_.reset_index(inplace=True)
genome_sizes_df_ = dict(zip(genome_sizes_df_["index"], genome_sizes_df_["genome_size"]))
genome_sizes_df_

In [ ]:
# G4Hunter
def complement(seq: str) -> str:
    seq = seq.upper()
    return seq if seq.count("G") >= seq.count("C") else seq.translate(str.maketrans("ACGT", "TGCA"))[::-1]

def complement(seq):
    seq = seq.upper()
    complement_seq = seq.translate(str.maketrans("ACGT", "TGCA"))[::-1]
    return min(seq, complement_seq)

g4_df.loc[:, "canonical"] = g4_df["sequence"].str.upper().apply(complement)
canonical_sequences = set(g4_df["canonical"])
# canonical_chm13 = set(g4_df["canonical_sequence"])
# canonical_chm13_rema = canonical_chm13 - canonical_sequences
# print(len(canonical_chm13_rema))

# Quadparser
regex_df.loc[:, "canonical"] = regex_df["sequence"].str.upper().apply(complement)
canonical_sequences_quad = set(regex_df["canonical"])
# canonical_chm13_quad = set(regex_df["canonical_sequence"])
# canonical_chm13_rema = canonical_chm13_quad - canonical_sequences_quad

## Motifs Across Haplotypes Shared

In [ ]:
sample_key = next(iter(processed_infiles))
seq_col = "sequence"
sequences_per_sample = {}
g4_densities = dict()

def complement_2(seq):
    seq = seq.upper()
    complement_seq = seq.translate(str.maketrans("ACGT", "TGCA"))[::-1]
    return min(seq, complement_seq)
for sample, (infile, haplotype, origin) in tqdm(processed_infiles.items()):
    df = pd.read_table(infile)
    seqs = df[seq_col].dropna().str.upper()
    seqs = seqs[~seqs.str.contains("N", na=False)]
    sequences_per_sample[sample] = set(seqs.apply(complement_2))

print(f"Loaded {len(sequences_per_sample)} haplotypes.")
print(f"Motifs per haplotype (first 3): { {k: len(v) for k, v in list(sequences_per_sample.items())[:3]} }")
g4_counter = defaultdict(int)
for sample, seqs in tqdm(sequences_per_sample.items()):
    for motif in seqs:
        g4_counter[motif] += 1
shared_df = (
    pd.Series(dict(g4_counter))
    .to_frame(name="haplotype_count")
    .reset_index()
    .rename(columns={"index": "sequence"})
    .assign(
        shared_proportion = lambda d: d["haplotype_count"] / len(sequences_per_sample),
        length            = lambda d: d["sequence"].str.len(),
    )
    .sort_values("haplotype_count", ascending=False)
    .reset_index(drop=True)
)

print(f"Total unique motifs: {shared_df.shape[0]}")
print(f"Present in all haplotypes: {(shared_df['haplotype_count'] == len(sequences_per_sample)).sum()}")
shared_df


In [ ]:
sample_key = next(iter(processed_infiles_quad))
seq_col = "sequence"
sequences_per_sample_quad = {}
g4_densities_quad = dict()
for sample, (infile, haplotype, origin) in tqdm(processed_infiles_quad.items()):
    df = pd.read_table(infile)
    seqs = df[seq_col].dropna().str.upper()
    seqs = seqs[~seqs.str.contains("N", na=False)]
    sequences_per_sample_quad[sample] = set(seqs.apply(complement_2))

print(f"Loaded {len(sequences_per_sample_quad)} haplotypes.")
print(f"Motifs per haplotype (first 3): { {k: len(v) for k, v in list(sequences_per_sample_quad.items())[:3]} }")
g4_counter_quad = defaultdict(int)
for sample, seqs in tqdm(sequences_per_sample_quad.items()):
    for motif in seqs:
        g4_counter_quad[motif] += 1
shared_quad_df = (
    pd.Series(dict(g4_counter_quad))
    .to_frame(name="haplotype_count")
    .reset_index()
    .rename(columns={"index": "sequence"})
    .assign(
        shared_proportion = lambda d: d["haplotype_count"] / len(sequences_per_sample_quad),
        length            = lambda d: d["sequence"].str.len(),
    )
    .sort_values("haplotype_count", ascending=False)
    .reset_index(drop=True)
)

print(f"Total unique motifs: {shared_quad_df.shape[0]}")
print(f"Present in all haplotypes: {(shared_quad_df['haplotype_count'] == len(sequences_per_sample_quad)).sum()}")
shared_quad_df


In [ ]:
def canonical_seq(seq):
    return seq if seq.count("G") >= seq.count("C") else seq.translate(str.maketrans("ACGT", "TGCA"))[::-1] 

shared_df.loc[:, "canonical"] = shared_df["sequence"].apply(canonical_seq)
shared_quad_df.loc[:, "canonical"] = shared_quad_df["sequence"].apply(canonical_seq)

In [ ]:
target = Path(os.getenv("WORK")).joinpath("g4_revisions/g4_t2t_revisions/G4_T2T/figures/haplotypes")
target_data = target.joinpath("data")
target_figures = target.joinpath("figures")
target.mkdir(exist_ok=True)
target_data.mkdir(exist_ok=True)
target_figures.mkdir(exist_ok=True)

In [ ]:
shared_df.to_csv(target_data.joinpath("shared_g4hunter_motifs.csv.gz"), index=False, compression="gzip")
shared_quad_df.to_csv(target_data.joinpath("shared_quadparser_motifs.csv.gz"), index=False, compression="gzip")

In [ ]:
# G4Hunter
g4_counter_grped_df = (
                    shared_df
                        .groupby("haplotype_count", as_index=False)\
                        .agg(sample_counts=("sequence", "count"))
                        .assign(log10_sample_counts=lambda ds: ds['sample_counts'].apply(lambda c: math.log(c, 10)))
        )
# Quadparser
g4_counter_grped_quad_df = (
                    shared_quad_df
                        .groupby("haplotype_count", as_index=False)\
                        .agg(sample_counts=("sequence", "count"))
                        .assign(log10_sample_counts=lambda ds: ds['sample_counts'].apply(lambda c: math.log(c, 10)))
        )
g4_counter_grped_df

In [ ]:
groups = [
          ("G4Hunter", g4_counter_grped_df),
          ("Quadparser", g4_counter_grped_quad_df)
          ]
for key, df in groups:
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(13, 5.6))
    plt.style.use('default')
    sns.barplot(data=df, 
                width=0.9,
                dodge=True, 
                x="haplotype_count", 
                y="sample_counts", 
                color="lightgray",
                edgecolor='black',
                ax=ax, 
                zorder=3, 
                capsize=.3)
    ax.grid(lw=0.6, alpha=0.4, zorder=0)
    ax.set_axisbelow(True)
    ax.set_ylabel("Occurrences")
    ax.set_xlabel("Total Haplotypes")
    ax.yaxis.label.set_size(22)
    ax.xaxis.label.set_size(22)
    ax.set_yscale("log", base=10)
    ax.tick_params(axis="both", labelsize=16)
    # Get the mid-top positions of each bar
    x = [p.get_x() + p.get_width() / 2 for p in ax.patches]  # X positions at the middle of each bar
    y = [p.get_height() for p in ax.patches]  # Heights of each bar

    # Plot a line connecting the mid-top of each bar
    # ax.set_xlim(-1.0, 94.3)

    for ind, label in enumerate(ax.get_xticklabels()):
        if ind % 3 == 0:  # every 10th label is kept
            label.set_visible(True)
        else:
            label.set_visible(False)
    fig.savefig(target_figures.joinpath(f"{key}_g4_unique_to_number_of_haplotypes.pdf"), 
                dpi=600, 
                transparent=True,
                format="pdf", 
                bbox_inches="tight")
    plt.show();

In [ ]:
groups = [
          ("G4Hunter", g4_counter_grped_df),
          ("Quadparser", g4_counter_grped_quad_df)
          ]

for key, df in groups:
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(13, 5.6))
    plt.style.use('default')
    sns.barplot(data=df, 
                width=0.9,
                dodge=True, 
                x="haplotype_count", 
                y="log10_sample_counts", 
                color="lightgray",
                edgecolor='black',
                ax=ax, 
                zorder=3, 
                capsize=.3)
    ax.grid(lw=0.6, alpha=0.4, zorder=0)
    ax.set_axisbelow(True)
    ax.set_ylabel(r"$\log_{10}(\mathrm{Occurrences})$", fontsize=16)
    ax.set_xlabel("Total Haplotypes")
    ax.yaxis.label.set_size(22)
    ax.xaxis.label.set_size(22)
    ax.tick_params(axis="both", labelsize=16)
    # Get the mid-top positions of each bar
    x = [p.get_x() + p.get_width() / 2 for p in ax.patches]  # X positions at the middle of each bar
    y = [p.get_height() for p in ax.patches]  # Heights of each bar

    # Plot a line connecting the mid-top of each bar
    # ax.set_xlim(-1.0, 94.3)

    for ind, label in enumerate(ax.get_xticklabels()):
        if ind % 3 == 0:  # every 10th label is kept
            label.set_visible(True)
        else:
            label.set_visible(False)
    fig.savefig(target_figures.joinpath(f"{key}_g4_unique_to_number_of_haplotypes.pdf"), 
                dpi=600, 
                transparent=True,
                format="pdf", 
                bbox_inches="tight")
    plt.show();

In [ ]:
import re
def calc_loop_ratio(seq):
    if pd.isna(seq):
        return np.nan
    g_runs = re.findall(r'G{2,}', seq.upper())
    g_run_bases = sum(len(g) for g in g_runs)
    if g_run_bases == 0:
        return np.nan
    loop_bases = len(seq) - g_run_bases
    return 1e2 * loop_bases / len(seq)

shared_df["loop_ratio"] = shared_df["sequence"].apply(calc_loop_ratio)
shared_quad_df["loop_ratio"] = shared_quad_df["sequence"].apply(calc_loop_ratio)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm

# G4Hunter
len_agg_g4_counter_g4hunter = shared_df.groupby("haplotype_count", as_index=False).agg(
                                                                                    avg_length=("length", "mean"),
                                                                                    total=("length", lambda ds: 1e2 * ds.count() / shared_df.shape[0]),
                                                                                    loop_ratio=("loop_ratio", "mean")
                                                                                )
len_agg_g4_counter_g4hunter.loc[:, "norm_loop_ratio"] = (len_agg_g4_counter_g4hunter["loop_ratio"] - len_agg_g4_counter_g4hunter["loop_ratio"].min())/ (len_agg_g4_counter_g4hunter["loop_ratio"].max() - len_agg_g4_counter_g4hunter["loop_ratio"].min())

# Quadparser
len_agg_g4_counter_quad = shared_quad_df.groupby("haplotype_count", as_index=False).agg(
                                                                                    avg_length=("length", "mean"),
                                                                                    total=("length", lambda ds: 1e2 * ds.count() / shared_quad_df.shape[0]),
                                                                                    loop_ratio=("loop_ratio", "mean")
                                                                                            )
len_agg_g4_counter_quad.loc[:, "norm_loop_ratio"] = (len_agg_g4_counter_quad["loop_ratio"] - len_agg_g4_counter_quad["loop_ratio"].min())/ (len_agg_g4_counter_quad["loop_ratio"].max() - len_agg_g4_counter_quad["loop_ratio"].min())

cmap = plt.cm.cividis
norm = mcolors.Normalize(vmin=0, vmax=1)
def value_to_single_color(val):
    rgba = cmap(norm(val))
    return rgba[0] * 255
    
len_agg_g4_counter_g4hunter.loc[:, "colors"] = len_agg_g4_counter_g4hunter["norm_loop_ratio"].apply(value_to_single_color)
len_agg_g4_counter_quad.loc[:, "colors"] = len_agg_g4_counter_quad["norm_loop_ratio"].apply(value_to_single_color)
len_agg_g4_counter_quad

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
import numpy as np
degree = 3
datasets = [
               ("G4Hunter", len_agg_g4_counter_g4hunter),
               ("Quadparser", len_agg_g4_counter_quad)
          ]
for db, df in datasets:
     poly = PolynomialFeatures(degree=degree, include_bias=True)
     X = df[["haplotype_count"]]
     y = df["avg_length"]

     X_poly = poly.fit_transform(X) # np.log10(X) # poly.fit_transform(X)
     model = LinearRegression()
     model.fit(X_poly, y)
     y_pred = model.predict(X_poly)
     fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(14.5, 6))

     ax.plot(X, y_pred,
               color="black",
               lw=2.0,
               linestyle='--',
               label="Regression Curve",
          )
     scatter = ax.scatter(
                    y=df["avg_length"],
                    x=df["haplotype_count"],
                    c=df["loop_ratio"],
                    edgecolor='black',
                    lw=1.0,
                    cmap="inferno",
                    s=70,
                    # s=len_agg_g4_counter_regex["total"],
                    )
     ax.legend(loc=0, prop={"size": 14})
     ax.grid(lw=0.4, alpha=0.6, zorder=0)
     ax.set_xlim(xmin=0, xmax=89)
     # sns.regplot(data=loop_agg_g4_counter_regex,
     #                 y="loop_ratio",
     #                 x="shared_proportion"
     #                )
     ax.tick_params(axis="y", labelsize=18)
     ax.tick_params(axis="x", labelsize=18)
     ax.set_ylabel("Average G4 Length")
     ax.set_xlabel("Total Haplotypes")
     ax.yaxis.label.set_size(24)
     ax.xaxis.label.set_size(24)

     # Add colorbar
     cbar = fig.colorbar(scatter, cmap="inferno", ax=ax)
     cbar.set_label('Average Loop Ratio (%)', size=24)  # Set the title for the colorbar
     cbar.ax.tick_params(labelsize=18)
     fig.savefig(target_figures.joinpath(f"avg_length_vs_haplotypes_shared_{db}_{degree}.pdf"), 
               bbox_inches="tight", 
               transparent=True,
               dpi=600, 
               format="pdf")
     plt.show();

## Total G4s per Haplotype

In [ ]:
from termcolor import colored
mode = "G4Hunter"
datasets = [
            ("G4Hunter", sequences_per_sample),
            ("Quadparser", sequences_per_sample_quad)
            ]
motif_db_counts = dict()
for database, samples in datasets:
    total_df = dict()
    for key, seq in tqdm(samples.items()):
        total_df[key] = len(seq)
    total_df = (
                    pd.Series(total_df)
                        .sort_values(ascending=False)
                        .to_frame(name="total_unique_motifs")
            )
    total_df['pop'] = total_df.index.map(lambda x: ancestries.get(x.split('#')[0], 'chm13v2'))
    # total['total'] = total.index.map(total_g4)
    # total['uniqueness'] = total['total_g4'].div(total['total'])
    dest = f"{target_data}/total_motifs_across_haplotypes_{database}.csv"
    total_df.to_csv(dest,
                                            sep=",",
                                            index=True,
                                            mode="w",
                                            header=True
                                            )
    if "Haplotypes" in total_df:
        total_df.drop(columns=["Haplotypes"], inplace=True)
    print(colored(f"Total motifs across haplotypes data have been saved at `{dest}`.", "green"))
    motif_db_counts[database] = total_df

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(13, 10))

panel_labels = ["A)", "B)"]

for ax, label, (key, df_raw) in zip(axes, panel_labels, motif_db_counts.items()):
    total_df = df_raw.copy()
    if "Haplotypes" not in total_df.columns:
        for cand in ("index", "level_0"):
            if cand in total_df.columns:
                total_df = total_df.rename(columns={cand: "Haplotypes"})
                break
        else:
            total_df = total_df.reset_index(names="Haplotypes")

    sns.barplot(data=total_df,
                width=0.9,
                dodge=False,
                x="Haplotypes",
                y="total_unique_motifs",
                hue="pop",
                linewidth=1.0,
                edgecolor='black',
                palette=superpopulation_colors,
                ax=ax,
                zorder=3,
                capsize=.3)
    ax.grid(lw=0.4, alpha=0.6, zorder=0)
    ax.set_axisbelow(True)
    ax.set_ylabel("G4 Occurrences")
    ax.set_xlabel("")
    ax.yaxis.label.set_size(22)
    ax.xaxis.label.set_size(22)
    ax.tick_params(axis="both", labelsize=18)
    ax.set_xticks([])
    ax.set_title(
        key, 
        fontsize=19, 
        fontweight="bold",
        y=1.05,
        bbox=dict(boxstyle="round,pad=0.4", 
                  facecolor="#D3D3D3", 
                  edgecolor="gray", 
                  linewidth=1.2),
    )
    ax.text(-0.06, 1.10, label,
            transform=ax.transAxes,
            fontsize=26,
            fontweight="bold",
            va="top",
            ha="left")
    ax.get_legend().remove()

handles, labels = axes[0].get_legend_handles_labels()
axes[0].legend(handles, labels,
               title="",
               prop={"size": 20},
               bbox_to_anchor=(1.02, 0.9),
               loc="upper left")

fig.tight_layout()
fig.subplots_adjust(hspace=0.30)

fig.savefig(target_figures.joinpath("total_g4_G4Hunter_Quadparser.pdf"),
            format="pdf",
            dpi=600,
            pad_inches=0.5,
            bbox_inches="tight",
            transparent=True)

In [ ]:
import re 
def extract_loop_ratio(sequence: str) -> float:
    """
    Calculates the loop ratio (percentage of non-G/C bases) in a motif sequence.
    
    Args:
        sequence (str): Motif sequence.
    
    Returns:
        float: Loop ratio as a percentage.
    """
    sequence = sequence.upper()
    leader = 'G' if sequence.count('G') >= sequence.count('C') else 'C'
    total_length = len(sequence)
    seq = re.sub("%s{2,}" % leader, "", sequence)
    loop_size = len(seq)
    return 1e2 * loop_size / total_length

In [ ]:
shared_df.loc[:, "length"] = shared_df["sequence"].apply(len)
shared_df.loc[:, "loop_ratio"] = shared_df["sequence"].apply(extract_loop_ratio)

shared_quad_df.loc[:, "length"] = shared_quad_df["sequence"].apply(len)
shared_quad_df.loc[:, "loop_ratio"] = shared_quad_df["sequence"].apply(extract_loop_ratio)
shared_df

## Shared with Reference

In [ ]:
from termcolor import colored 

def shared_with_reference_genome(g4_df, sequences_per_sample: dict, mode: str = "G4Hunter") -> pd.DataFrame:
        """
        For each haplotype, extracts the percentage of motifs from the reference genome found in the haplotype.
        Removes non-autosomal chromosomes for this analysis.
        
        Args:
            sequences_per_sample (dict): Dictionary mapping sample IDs to sets of motif sequences.
        
        Returns:
            pd.DataFrame: DataFrame with sharing statistics per haplotype and chromosome.
        """
        # fetch autosomal motifs
        autosomal_motifs = g4_df.query("seqID != 'chrX' & seqID != 'chrY'").dropna(subset=['seqID'])
        # fetch autosomal chromosomes
        sequence_ids = set(autosomal_motifs["seqID"].unique())
        total_sequence_ids = len(sequence_ids)
        print(f"Total autosomal motifs: {autosomal_motifs.shape[0]}.")
        print(f"Total autosomal chromosomes: {total_sequence_ids}.")

        chm13v2_sequences_specific_seqID = dict()
        total_ref = dict()
        for seqID in tqdm(sequence_ids, total=total_sequence_ids):
            chm13v2_sequences_specific_seqID[seqID] = set(autosomal_motifs[autosomal_motifs["seqID"] == seqID]["canonical"])
            total_ref[seqID] = len(chm13v2_sequences_specific_seqID[seqID])

        common_with_reference_specific = defaultdict(list)
        for sample in tqdm(sequences_per_sample):
            seq = sequences_per_sample[sample]
            for seqID in sequence_ids:
                chm13v2_sequences_specific = chm13v2_sequences_specific_seqID[seqID]
                mutual = seq.intersection(chm13v2_sequences_specific)
                shared = seq.union(chm13v2_sequences_specific)
                jaccard_index = len(mutual) / len(shared)
                
                common_with_reference_specific["haplotype"].append(sample)
                common_with_reference_specific["jaccard"].append(jaccard_index)
                common_with_reference_specific["union"].append(len(shared))
                common_with_reference_specific["total_ref"].append(total_ref[seqID])
                common_with_reference_specific["pop"].append(ancestries[sample.split('#')[0]])
                common_with_reference_specific["mutual"].append(len(mutual))
                common_with_reference_specific["total"].append(len(seq))
                common_with_reference_specific["seqID"].append(seqID)

        common_with_reference_specific = pd.DataFrame(common_with_reference_specific)
        unique_g4_motifs_per_seqID = autosomal_motifs.groupby("seqID").agg({"canonical": "nunique"})["canonical"].to_dict()
        common_with_reference_specific.loc[:, "unique_ref_motifs"] = common_with_reference_specific["seqID"].map(unique_g4_motifs_per_seqID)
        common_with_reference_specific.loc[:, "shared_with_ref_perc"] = 1e2 * common_with_reference_specific["mutual"] / common_with_reference_specific["total_ref"]
        dest = f"{target_data}/common_with_reference_specific_motifs_{mode}.csv"
        common_with_reference_specific.to_csv(dest,
                                                sep=",",
                                                index=False,
                                                mode="w",
                                                header=True
                                                )
        print(colored(f"Motifs shared with reference genome data have been saved at `{dest}`.", "green"))
        return common_with_reference_specific 
    

In [ ]:
# G4Hunter
common_with_reference_specific_g4 = shared_with_reference_genome(g4_df, 
                            sequences_per_sample, 
                            mode="G4Hunter")
# Quadparser
common_with_reference_specific_regex = shared_with_reference_genome(regex_df, 
                                        sequences_per_sample_quad, 
                                        mode="Quadparser")

# common_with_reference_specific_regex = shared_with_reference_genome(regex_df, sequences_per_sample, mode="Quadparser")
common_with_reference_specific_g4    

In [ ]:
haplotype_colors = {
                    "paternal": "#d687d1",
                    "maternal": "#f5eba2"
}
sex_colors = {
               "male": "#455aba",
              "female": "#92d4b7"
             }

superpopulation_colors = {
              "EAS": "#ed1a76",
              "AMR": "#fcba03",
              "SAS": "#036bfc",
              "AFR": "#169e87"
             }

In [ ]:
# Quadparser
MODE = "Quadparser"
common_with_reference_specific_quad_pivot = common_with_reference_specific_regex.pivot(
                                                                        index="haplotype", 
                                                                        columns="seqID", 
                                                                        values="shared_with_ref_perc")

row_cols = common_with_reference_specific_quad_pivot.index.map(lambda x: superpopulation_colors[ancestries[x.split('#')[0]]])
common_with_reference_specific_quad_pivot.drop(columns=["chrM"], inplace=True)
for col in common_with_reference_specific_quad_pivot.index.tolist():
    col = col.split("#")[0]
    if col not in ancestries:
        print(col)
        
row_colors = common_with_reference_specific_quad_pivot.index.map(lambda x: ancestries[x.split("#")[0]]).map(superpopulation_colors)
row_colors_x = common_with_reference_specific_quad_pivot.index.map(lambda x: x.split("#")[1]).map(haplotype_colors)
row_colors_y = common_with_reference_specific_quad_pivot.index.map(lambda x: sex_pop_meta[x.split("#")[0]]).map(sex_colors)
cg = sns.clustermap(
                data=common_with_reference_specific_quad_pivot, 
                cmap="coolwarm",
                row_colors=[row_colors_y, row_colors],
                cbar_pos=(0.07, 0.15, 0.03, 0.6),
                figsize=(13, 6.5),
                row_cluster=True,
                col_cluster=True,
            )
cg.ax_row_dendrogram.set_visible(False)
cg.ax_col_dendrogram.set_visible(False) 
ax = cg.ax_heatmap

ax.set_xticklabels(cg.ax_heatmap.get_xmajorticklabels(), fontsize = 16)
ax.set_ylabel('')
ax.tick_params(axis="y", rotation=0, labelsize=16)

cg.cax.set_ylabel("Shared G4 Motifs (%)", fontsize=18)
cg.cax.tick_params(labelsize=14)

ax.set_xlabel('')
ax.set_ylabel("Haplotypes")
ax.yaxis.label.set_size(20)
ax.set_yticks([])

for i in range(len(common_with_reference_specific_quad_pivot.columns) + 2):  # +1 to include the last edge
    ax.axvline(i, color="black", lw=0.8)
    # ax.axhline(i, color="black", lw=0.8)
ax.axhline(0, color="black", lw=0.8)
ax.axhline(len(common_with_reference_specific_quad_pivot.index), 
           color="black", 
           lw=0.8)

row_order = cg.dendrogram_row.reordered_ind
col_order = cg.dendrogram_col.reordered_ind
fig = plt.gcf()
fig.savefig(f"{target_figures}/clustermap_shared_perc_with_reference_{MODE}.pdf", 
            bbox_inches="tight",
            dpi=600,
           format="pdf");


In [ ]:
# G4Hunter
MODE = "G$Hunter"
common_with_reference_specific_pivot = common_with_reference_specific_g4.pivot(index="haplotype", 
                                                                            columns="seqID", 
                                                                            values="shared_with_ref_perc")

row_cols = common_with_reference_specific_pivot.index.map(lambda x: superpopulation_colors[ancestries[x.split('#')[0]]])
common_with_reference_specific_pivot.drop(columns=["chrM"], inplace=True)
for col in common_with_reference_specific_pivot.index.tolist():
    col = col.split("#")[0]
    if col not in ancestries:
        print(col)
row_colors = common_with_reference_specific_pivot.index.map(lambda x: ancestries[x.split("#")[0]]).map(superpopulation_colors)
row_colors_x = common_with_reference_specific_pivot.index.map(lambda x: x.split("#")[1]).map(haplotype_colors)
row_colors_y = common_with_reference_specific_pivot.index.map(lambda x: sex_pop_meta[x.split("#")[0]]).map(sex_colors)

cg = sns.clustermap(
                data=common_with_reference_specific_pivot, 
                cmap="coolwarm",
                row_colors=[row_colors_y, row_colors],
                cbar_pos=(0.07, 0.15, 0.03, 0.6),
                figsize=(13, 6.5),
                row_cluster=True,
                col_cluster=True,
            )
cg.ax_row_dendrogram.set_visible(False)
cg.ax_col_dendrogram.set_visible(False) 
ax = cg.ax_heatmap

ax.set_xticklabels(cg.ax_heatmap.get_xmajorticklabels(), fontsize = 16)
ax.set_ylabel('')
ax.tick_params(axis="y", rotation=0, labelsize=16)

cg.cax.set_ylabel("Shared G4 Motifs (%)", fontsize=18)
cg.cax.tick_params(labelsize=14)

ax.set_xlabel('')
ax.set_ylabel("Haplotypes")
ax.yaxis.label.set_size(20)
ax.set_yticks([])

for i in range(len(common_with_reference_specific_pivot.columns) + 2):  # +1 to include the last edge
    ax.axvline(i, color="black", lw=0.8)
    # ax.axhline(i, color="black", lw=0.8)
ax.axhline(0, color="black", lw=0.8)
ax.axhline(len(common_with_reference_specific_pivot.index), 
           color="black", 
           lw=0.8)

row_order = cg.dendrogram_row.reordered_ind
col_order = cg.dendrogram_col.reordered_ind
fig = plt.gcf()
fig.savefig(f"{target_figures}/clustermap_shared_perc_with_reference_{MODE}.pdf", 
            bbox_inches="tight",
            dpi=600,
           format="pdf");

## Unique Motifs per Haplotype

In [ ]:
databases = [
            ("G4Hunter", sequences_per_sample),
            ("Quadparser", sequences_per_sample_quad)
            ]

unique_db = dict()
for db, samples in databases:
    unique_sequences = defaultdict(int)
    unique_sequences_sets = {} 
    for sample_A, seqs_A in tqdm(samples.items()):
        remaining = seqs_A.copy()          
        # don't mutate the original set
        for sample_B, seqs_B in samples.items():
            if sample_A == sample_B:
                continue
            remaining -= seqs_B            # subtract cumulatively — correct
        unique_sequences[sample_A] = len(remaining)
        unique_sequences_sets[sample_A] = remaining
        
    afr_unique_seqs = set.union(*[
        v for s, v in unique_sequences_sets.items() 
        if ancestries[s.split("#")[0]] == "AFR"
    ])

    non_afr_unique_seqs = set.union(*[
        v for s, v in unique_sequences_sets.items() 
        if ancestries[s.split("#")[0]] != "AFR"
    ])

    print(f"AFR-unique:     {len(afr_unique_seqs)}")
    print(f"Non-AFR-unique: {len(non_afr_unique_seqs)}")

    unique_df = (
        pd.Series(dict(unique_sequences))
        .to_frame(name="unique_motifs")
        .reset_index()
        .rename(columns={"index": "haplotype"})
        .sort_values("unique_motifs", ascending=False)
        .reset_index(drop=True)
    )

    print(f"Total haplotypes: {len(unique_df)}")
    print(f"Haplotypes with 0 unique motifs: {(unique_df['unique_motifs'] == 0).sum()}")
    unique_df.to_csv(f"{target_data}/unique_motifs_per_haplotype_{db}.csv.gz",
                    compression="gzip", 
                 sep=",", 
                 mode="w", 
                 index=False)
    unique_db[db] = unique_df

In [ ]:
AFR_samples    = {s for s in sequences_per_sample if ancestries[s.split("#")[0]] == "AFR"}
non_AFR_samples = {s for s in sequences_per_sample if ancestries[s.split("#")[0]] != "AFR"}

afr_all     = set.union(*[sequences_per_sample[s] for s in AFR_samples])
non_afr_all = set.union(*[sequences_per_sample[s] for s in non_AFR_samples])

afr_unique_seqs     = afr_all - non_afr_all
non_afr_unique_seqs = non_afr_all - afr_all

print(f"AFR-unique:     {len(afr_unique_seqs)}")
print(f"Non-AFR-unique: {len(non_afr_unique_seqs)}")

In [ ]:
def count_gruns(seq, min_g=3, max_g=None):
    motif = "G" if seq.count("G") >= seq.count("C") else "C"
    pattern = rf"{motif}{{{min_g},}}"
    return len(re.findall(pattern, seq.upper()))

non_afr_unique_seqs = pd.Series(list(non_afr_unique_seqs)).to_frame(name="sequence")
non_afr_unique_seqs.loc[:, "length"] = non_afr_unique_seqs["sequence"].str.len()
non_afr_unique_seqs.loc[:, "loop_ratio"] = non_afr_unique_seqs["sequence"].apply(extract_loop_ratio)
non_afr_unique_seqs.loc[:, "g_run_count"] = non_afr_unique_seqs["sequence"].apply(count_gruns)
non_afr_unique_seqs.to_csv(f"{target_data}/non_AFR_unique_motifs_G4HUNTER.csv.gz", compression="gzip", sep=",", mode="w", index=False) 

afr_unique_seqs = pd.Series(list(afr_unique_seqs)).to_frame(name="sequence")
afr_unique_seqs.loc[:, "length"] = afr_unique_seqs["sequence"].str.len()
afr_unique_seqs.loc[:, "loop_ratio"] = afr_unique_seqs["sequence"].apply(extract_loop_ratio)
afr_unique_seqs.loc[:, "g_run_count"] = afr_unique_seqs["sequence"].apply(count_gruns)
afr_unique_seqs.to_csv(f"{target_data}/AFR_unique_motifs_G4HUNTER.csv.gz", compression="gzip", sep=",", mode="w", index=False)
afr_unique_seqs

In [ ]:
unique_to_non_afr_motifs = non_afr_unique_seqs.sort_values("length", ascending=True).query("g_run_count >= 3").query("length <= 30")
unique_to_non_afr_motifs_set = set(unique_to_non_afr_motifs["sequence"])
unique_to_non_afr_motifs

In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig_dir = target
fig_dir.mkdir(exist_ok=True)
haplotype_colors = {"paternal": "#d687d1",
                     "maternal": "#f5eba2"
                     }
sex_colors = {
               "XY": "#455aba",
              "XX": "#92d4b7"
             }

superpopulation_colors = {
              "EAS": "#ed1a76",
              "AMR": "#fcba03",
              "SAS": "#036bfc",
              "AFR": "#169e87"
             }
legend_configs = {
    "legend_haplotype": haplotype_colors,
    "legend_sex": sex_colors,
    "legend_superpopulation": superpopulation_colors,
}

for fname, color_dict in legend_configs.items():
    handles = [mpatches.Patch(facecolor=c, label=l) for l, c in color_dict.items()]
    fig, ax = plt.subplots(figsize=(2, len(color_dict) * 0.5))
    ax.axis("off")
    if fname == "legend_haplotype":
        ax.legend(
        handles=handles,
        loc="center",
        fontsize=16,
        title="Haplotype",
        frameon=True,
        fancybox=True,
        shadow=True,
        handlelength=1.5,title_fontsize=16,


        handleheight=1.5,
        )
    else:
            ax.legend(
        handles=handles,
        loc="center",
        fontsize=16,
        frameon=True,
        fancybox=True,
        shadow=True,
        title_fontsize=24,
        handlelength=1.5,
        handleheight=1.5,
        )
    fig.savefig(target / f"{fname}.pdf", dpi=600, bbox_inches="tight", transparent=True)
    plt.show()


In [ ]:
sex_colors = {"male": "#455aba", 
              "female": "#92d4b7"}
# # 
for MODE, unique_df in unique_db.items():
    unique_df.loc[:, "population"] = unique_df["haplotype"].map(lambda x: ancestries.get(x.split("#")[0], "."))
    sex_colors_x = unique_df["haplotype"].apply(lambda x: sex_colors[sex_pop_meta[x.split("#")[0]]]).tolist()
    haplotype_colors_x = unique_df["haplotype"].apply(lambda x: haplotype_colors[x.split("#")[1]]).tolist()
    sex_colors_x = unique_df["haplotype"].apply(lambda x: sex_colors[sex_pop_meta[x.split("#")[0]]]).tolist()
    haplotype_colors_x[:5]
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(13, 6))
    plt.style.use('default')
    sns.barplot(data=unique_df, 
                width=0.9, 
                dodge=False, 
                x="haplotype", 
                y="unique_motifs", 
                hue="population",
                palette=superpopulation_colors,
                ax=ax, 
                zorder=3, 
                capsize=.3)
    ax.grid(lw=0.6, alpha=0.4, zorder=0)
    ax.set_axisbelow(True)
    ax.set_ylabel("Unique G4s")
    ax.set_xlabel("Haplotypes")
    ax.yaxis.label.set_size(22)
    ax.xaxis.label.set_size(22)
    ax.tick_params(axis="both", labelsize=18)
    ax.legend(title="", prop={"size": 20})

    xticks = ax.get_xticks()
    for x, (sex_color, superpop_color) in zip(xticks, zip(sex_colors_x, haplotype_colors_x)):
        # Top rectangle (closer to bars)
        ax.add_patch(plt.Rectangle((x - 0.45, -0.06 * max(unique_df["unique_motifs"])), 
                                    0.95, 
                                0.06 * max(unique_df["unique_motifs"]),
                                    color=superpop_color, 
                                    transform=ax.transData, 
                                    clip_on=False))
        
        # Bottom rectangle (further down)
        ax.add_patch(plt.Rectangle((x - 0.45, -0.12 * max(unique_df["unique_motifs"])), 
                                    0.95, 
                                0.06 * max(unique_df["unique_motifs"]),
                                    color=sex_color, 
                                    transform=ax.transData, 
                                    clip_on=False))

    ax.set_xticks([])
    # ax.set_ylim(bottom=-0.08 * max(unique_df["unique_motifs"]))
    ax.xaxis.set_label_coords(0.5, -0.13)
    fig.savefig(target_figures.joinpath(f"{MODE}_unique_g4hunter_to_each_haplotype.with_haplotype_lineage_origin.pdf"), 
                dpi=600, 
                transparent=True,
                format="pdf", 
                bbox_inches="tight");
    plt.show()

### Unique Motifs per Assembly

In [ ]:
per_assembly = defaultdict(set)
for sample_A in tqdm(sequences_per_sample):
    seq = sequences_per_sample[sample_A]
    s_A = sample_A.split('#')[0]
    per_assembly[s_A] = per_assembly[s_A].union(seq)
len(per_assembly)

# Quadparser
per_assembly_quad = defaultdict(set)
for sample_A in tqdm(sequences_per_sample_quad):
    seq = sequences_per_sample_quad[sample_A]
    s_A = sample_A.split('#')[0]
    per_assembly_quad[s_A] = per_assembly_quad[s_A].union(seq)
len(per_assembly_quad)

In [ ]:
# G4Hunter
unique_sequences_per_assembly = defaultdict(int)
unique_sequences_sets_per_assembly = {} 
for sample_A, seqs_A in tqdm(per_assembly.items()):
    remaining = seqs_A.copy()          
    # don't mutate the original set
    for sample_B, seqs_B in per_assembly.items():
        if sample_A == sample_B:
            continue
        remaining -= seqs_B            # subtract cumulatively — correct
    unique_sequences_per_assembly[sample_A] = len(remaining)
    unique_sequences_sets_per_assembly[sample_A] = remaining
    
afr_unique_seqs_per_assembly = set.union(*[
    v for s, v in unique_sequences_sets_per_assembly.items() 
    if ancestries[s.split("#")[0]] == "AFR"
])

non_afr_unique_seqs_per_assembly = set.union(*[
    v for s, v in unique_sequences_sets_per_assembly.items() 
    if ancestries[s.split("#")[0]] != "AFR"
])

# Quadparser
unique_sequences_per_assembly_quad = defaultdict(int)
unique_sequences_sets_per_assembly_quad = {} 
for sample_A, seqs_A in tqdm(per_assembly_quad.items()):
    remaining = seqs_A.copy()          
    # don't mutate the original set
    for sample_B, seqs_B in per_assembly_quad.items():
        if sample_A == sample_B:
            continue
        remaining -= seqs_B            # subtract cumulatively — correct
    unique_sequences_per_assembly_quad[sample_A] = len(remaining)
    unique_sequences_sets_per_assembly_quad[sample_A] = remaining
    
afr_unique_seqs_per_assembly_quad = set.union(*[
    v for s, v in unique_sequences_sets_per_assembly_quad.items() 
    if ancestries[s.split("#")[0]] == "AFR"
])

non_afr_unique_seqs_per_assembly_quad = set.union(*[
    v for s, v in unique_sequences_sets_per_assembly_quad.items() 
    if ancestries[s.split("#")[0]] != "AFR"
])

In [ ]:
# G4Hunter
unique_assembly_df = (
    pd.Series(unique_sequences_per_assembly).to_frame("unique_motifs")
    .reset_index().rename(columns={"index": "assembly"})
    .sort_values("unique_motifs", ascending=False)
    .reset_index(drop=True)
)

print(f"Total assemblies: {len(unique_assembly_df)}")
print(f"Assemblies with 0 unique motifs: {(unique_assembly_df['unique_motifs'] == 0).sum()}")
unique_assembly_df['population'] = unique_assembly_df['assembly'].apply(lambda x: ancestries[x])
unique_assembly_df = unique_assembly_df.sort_values(by=['unique_motifs'], ascending=False).reset_index(drop=True)
# unique_assembly_df['sex'] = unique_assembly_df['Haplotype'].str.split('.', expand=True)[1]
unique_assembly_df

# Quadparser
unique_assembly_quad_df = (
    pd.Series(unique_sequences_per_assembly_quad).to_frame("unique_motifs")
    .reset_index().rename(columns={"index": "assembly"})
    .sort_values("unique_motifs", ascending=False)
    .reset_index(drop=True)
)

print(f"Total assemblies: {len(unique_assembly_quad_df)}")
print(f"Assemblies with 0 unique motifs: {(unique_assembly_quad_df['unique_motifs'] == 0).sum()}")
unique_assembly_quad_df['population'] = unique_assembly_quad_df['assembly'].apply(lambda x: ancestries[x])
unique_assembly_quad_df = unique_assembly_quad_df.sort_values(by=['unique_motifs'], ascending=False).reset_index(drop=True)
# unique_assembly_df['sex'] = unique_assembly_df['Haplotype'].str.split('.', expand=True)[1]
unique_assembly_quad_df

In [ ]:
# G4Hunter
MODE = "G4Hunter"
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(13, 6))
sex_colors_x = unique_assembly_df["assembly"].apply(lambda x: sex_colors[sex_pop_meta[x]]).tolist()
sns.barplot(data=unique_assembly_df, 
            width=0.9, 
            dodge=False, 
            x="assembly", 
            y="unique_motifs", 
            hue="population",
            palette=superpopulation_colors,
            ax=ax, 
            zorder=3, 
            capsize=.3)
ax.grid(lw=0.6, alpha=0.4, zorder=0)
ax.set_axisbelow(True)
ax.set_ylabel("Unique G4 Motifs")
ax.set_xlabel("")
ax.yaxis.label.set_size(22)
ax.xaxis.label.set_size(22)
ax.tick_params(axis="both", labelsize=18)
ax.legend(title="", prop={"size": 20})


xticks = ax.get_xticks()
for x, sex_color in zip(xticks, sex_colors_x):
    ax.add_patch(plt.Rectangle((x - 0.45, -0.06 * max(unique_assembly_df["unique_motifs"])), 
                                1.0, 
                               0.06 * max(unique_assembly_df["unique_motifs"]),
                                color=sex_color, 
                                transform=ax.transData, 
                                clip_on=False))
ax.set_xticks([])
ax.xaxis.set_label_coords(0.5, -0.10)
fig.savefig(target_figures.joinpath(f"{MODE}_unique_g4hunter_to_each_haplotype_no_sex_diff.pdf"), 
            format="pdf", 
            dpi=600, 
            bbox_inches="tight")

In [ ]:
# Quadparser 
MODE = "Quadparser"
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(13, 6))
sex_colors_x = unique_assembly_quad_df["assembly"].apply(lambda x: sex_colors[sex_pop_meta[x]]).tolist()
sns.barplot(data=unique_assembly_quad_df, 
            width=0.9, 
            dodge=False, 
            x="assembly", 
            y="unique_motifs", 
            hue="population",
            palette=superpopulation_colors,
            ax=ax, 
            zorder=3, 
            capsize=.3)
ax.grid(lw=0.6, alpha=0.4, zorder=0)
ax.set_axisbelow(True)
ax.set_ylabel("Unique G4 Motifs")
ax.set_xlabel("")
ax.yaxis.label.set_size(22)
ax.xaxis.label.set_size(22)
ax.tick_params(axis="both", labelsize=18)
ax.legend(title="", prop={"size": 20})


xticks = ax.get_xticks()
for x, sex_color in zip(xticks, sex_colors_x):
    ax.add_patch(plt.Rectangle((x - 0.45, -0.06 * max(unique_assembly_quad_df["unique_motifs"])), 
                                1.0, 
                               0.06 * max(unique_assembly_quad_df["unique_motifs"]),
                                color=sex_color, 
                                transform=ax.transData, 
                                clip_on=False))
ax.set_xticks([])
ax.xaxis.set_label_coords(0.5, -0.10)
fig.savefig(target_figures.joinpath(f"{MODE}_unique_g4hunter_to_each_haplotype_no_sex_diff.pdf"), 
            format="pdf", 
            dpi=600, 
            bbox_inches="tight")

## African Genomes Harbor more Unique Motifs

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from tqdm import tqdm

N_DRAWS = 100
rng = np.random.default_rng(0)

def encode(samples):
    """{haplotype: set(seq)} -> ({haplotype: np.array(int32 ids)}, n_ids)"""
    seq_to_id = {}
    hap_ids = {}
    for h, seqs in samples.items():
        ids = np.empty(len(seqs), dtype=np.int32)
        for i, s in enumerate(seqs):
            ids[i] = seq_to_id.setdefault(s, len(seq_to_id))
        hap_ids[h] = ids
    return hap_ids, len(seq_to_id)


def private_counts(hap_ids, n_ids, panel):
    """number of sequences private to each haplotype, within `panel` only"""
    occ = np.zeros(n_ids, dtype=np.int16)
    for h in panel:
        occ[hap_ids[h]] += 1          # sets => no dupes within a haplotype
    singletons = occ == 1
    return {h: int(singletons[hap_ids[h]].sum()) for h in panel}


rows = []
for db, samples in databases:
    hap_ids, n_ids = encode(samples)
    group_of = {h: ("AFR" if ancestries[h.split("#")[0]] == "AFR" else "non-AFR")
                for h in samples}

    afr_haps = [h for h in samples if group_of[h] == "AFR"]
    non_haps = [h for h in samples if group_of[h] == "non-AFR"]
    n = min(len(afr_haps), len(non_haps))
    print(f"{db}: n_AFR={len(afr_haps)}, n_nonAFR={len(non_haps)} -> {n} per group per draw")

    for draw in tqdm(range(N_DRAWS), desc=db):
        panel = [*rng.choice(afr_haps, n, replace=False),
                 *rng.choice(non_haps, n, replace=False)]
        for h, c in private_counts(hap_ids, n_ids, panel).items():
            rows.append({
                "method": db,
                "draw": draw,
                "haplotype": h,
                "afr_group": group_of[h],
                "unique_motifs": c,
                "frac_unique": c / len(hap_ids[h]),
            })

balanced_df = pd.DataFrame(rows)
balanced_df.to_csv(f"{target_data}/balanced_subsample_unique_motifs.csv.gz", sep="\t", compression="gzip", index=False)
balanced_df

In [ ]:
# import numpy as np
# def get_base_score(line: str) -> tuple[str, list[int]]:
#     item, score_list = 0, []
#     # calcule le item de chaque base et la stock dans score_list
#     while (item < len(line)):
#         if (item < len(line) and (line[item]=="G" or line[item]=="g")):
#             score_list.append(1)
#             if(item+1< len(line) and (line[item+1]=="G" or line[item+1]=="g")):
#                 score_list[item]=2
#                 score_list.append(2)
#                 if (item+2< len(line) and (line[item+2]=="G" or line[item+2]=="g")):
#                     score_list[item+1]=3
#                     score_list[item]=3
#                     score_list.append(3)
#                     if (item+3< len(line) and (line[item+3]=="G" or line[item+3]=="g")):
#                         score_list[item]=4
#                         score_list[item+1]=4
#                         score_list[item+2]=4
#                         score_list.append(4)
#                         item=item+1
#                     item=item+1
#                 item=item+1
#             item=item+1
#             while(item < len(line) and (line[item]=="G" or line[item]=="g")):
#                     score_list.append(4)
#                     item=item+1

#         elif (item < len(line) and line[item]!="G" and line[item]!="g" and line[item]!= "C" and line[item]!="c" ):
#                     score_list.append(0)
#                     item=item+1
            
#         elif(item < len(line) and (line[item]=="C" or line[item]=="c")):
#             score_list.append(-1)
#             if(item+1< len(line) and (line[item+1]=="C" or line[item+1]=="c" )):
#                 score_list[item]=-2
#                 score_list.append(-2)
#                 if (item+2< len(line) and (line[item+2]=="C" or line[item+2]=="c" )):
#                     score_list[item+1]=-3
#                     score_list[item]=-3
#                     score_list.append(-3)
#                     if (item+3< len(line) and (line[item+3]=="C" or line[item+3]=="c"  )):
#                         score_list[item]=-4
#                         score_list[item+1]=-4
#                         score_list[item+2]=-4
#                         score_list.append(-4)
#                         item=item+1
#                     item=item+1   
#                 item=item+1
#             item=item+1
#             while(item < len(line) and (line[item]=="C" or line[item]=="c")):
#                 score_list.append(-4)
#                 item=item+1
#         else:
#                 item=item+1 
#     # return line, score_list
#     return np.mean(score_list)

In [ ]:
summary = []
for (db, draw), g in balanced_df.groupby(["method", "draw"]):
    a = g.loc[g.afr_group == "AFR", "unique_motifs"].values
    b = g.loc[g.afr_group == "non-AFR", "unique_motifs"].values
    u, p = mannwhitneyu(a, b, alternative="two-sided")
    summary.append({
        "method": db,
        "draw": draw,
        "median_AFR": np.median(a),
        "median_nonAFR": np.median(b),
        "diff": np.median(a) - np.median(b),
        "cliffs_delta": 2 * u / (len(a) * len(b)) - 1,
        "p": p,
    })

summary_df = pd.DataFrame(summary)

for db, g in summary_df.groupby("method"):
    lo, hi = np.percentile(g["diff"], [2.5, 97.5])
    d_lo, d_hi = np.percentile(g["cliffs_delta"], [2.5, 97.5])
    print(f"\n{db}")
    print(f"  median diff (AFR - non-AFR): {g['diff'].median():.1f}  [{lo:.1f}, {hi:.1f}]")
    print(f"  Cliff's delta:               {g['cliffs_delta'].median():.3f}  [{d_lo:.3f}, {d_hi:.3f}]")
    print(f"  draws with p < 0.05:         {(g['p'] < 0.05).mean():.0%}")


In [ ]:
for db, g in balanced_df[balanced_df.draw == 0].groupby("method"):
    a = g.loc[g.afr_group == "AFR", "frac_unique"]
    b = g.loc[g.afr_group == "non-AFR", "frac_unique"]
    u, p = mannwhitneyu(a, b, alternative="two-sided")
    print(f"{db}: AFR={a.median():.4f}  non-AFR={b.median():.4f}  "
          f"delta={2*u/(len(a)*len(b))-1:.3f}  p={p:.2e}")


In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4.5))

plot_params = dict(
    data=balanced_df,
    x="method",
    y="unique_motifs",
    hue="afr_group",
    order=[db for db, _ in databases],
    hue_order=["AFR", "non-AFR"],
)

sns.boxplot(
    **plot_params,
    palette={"AFR": superpopulation_colors["AFR"], 
             "non-AFR": "#aaaaaa"},
    width=0.5,
    linewidth=1.5,
    flierprops=dict(marker="o", markersize=2, alpha=0.3),
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("Unique G4s (balanced panels)", fontsize=20)
ax.tick_params(axis="both", labelsize=16)
ax.grid(lw=0.4, alpha=0.6, zorder=0)
ax.set_axisbelow(True)
ax.legend(title="", fontsize=13, frameon=False)
sns.despine()
plt.tight_layout()

fig.savefig(target_figures / "unique_g4_AFR_vs_nonAFR_balanced.pdf",
            format="pdf", 
            transparent=True, 
            bbox_inches="tight", 
            dpi=600)
plt.show()


In [ ]:
unique_assembly_df.loc[:, "sex"] = unique_assembly_df["assembly"].map(lambda x: sex_pop_meta[x.split("#")[0]])
unique_assembly_df

In [ ]:
from scipy.stats import kruskal
groups = [grp["unique_motifs"].values for _, grp in unique_df.groupby("population")]
h_stat, p_kw = kruskal(*groups)
print(f"Kruskal-Wallis: H={h_stat:.3f}, p={p_kw:.2e}")

from scipy.stats import kruskal
groups = [grp["unique_motifs"].values for _, grp in unique_assembly_df.groupby("population")]
h_stat, p_kw = kruskal(*groups)
print(f"Kruskal-Wallis: H={h_stat:.3f}, p={p_kw:.2e}")

In [ ]:
unique_df_quad = unique_db["Quadparser"].copy()
unique_df = unique_db["G4Hunter"].copy()
unique_df.loc[:, "method"] = "G4Hunter"
unique_df_quad.loc[:, "method"] = "Quadparser"
unique_df_combined = pd.concat([unique_df, unique_df_quad], axis=0, ignore_index=True
                               )
unique_df_combined.loc[:, "population"] = unique_df_combined["haplotype"].map(lambda x: ancestries.get(x.split("#")[0]))
unique_df_combined.loc[:, "origin"] = unique_df_combined["haplotype"].map(lambda x: x.split("#")[1])
unique_df_combined.loc[:, "origin"] = unique_df_combined["origin"].str.capitalize()
unique_df_combined.loc[:, "sex"] = unique_df_combined["haplotype"].map(lambda x: sex_pop_meta[x.split("#")[0]])
# unique_df_combined.loc[:, "genome_size"] = unique_df_combined["haplotype"].map(genome_sizes_df_)
# unique_df_combined.loc[:, "unique_motifs_per_mb"] = 1e6 * unique_df_combined["unique_motifs"] / unique_df_combined["genome_size"]
unique_df_combined

In [ ]:
genome_sizes_df_per_assembly = defaultdict(int)
for key, genome_size in genome_sizes_df_.items():
    genome_sizes_df_per_assembly[key.split("#")[0]] += genome_size
genome_sizes_df_per_assembly

In [ ]:
unique_assembly_df.loc[:, "method"] = "G4Hunter"
unique_assembly_quad_df.loc[:, "method"] = "Quadparser"
unique_assembly_df_combined = pd.concat([unique_assembly_df, unique_assembly_quad_df], axis=0, ignore_index=True)
unique_assembly_df_combined.loc[:, "sex"] = unique_assembly_df_combined["assembly"].map(lambda x: sex_pop_meta[x])
# unique_assembly_df_combined.loc[:, "genome_size"] = unique_assembly_df_combined["assembly"].map(genome_sizes_df_per_assembly)
# unique_assembly_df_combined.loc[:, "unique_motifs_per_mb"] = 1e6 * unique_assembly_df_combined["unique_motifs"] / unique_assembly_df_combined["genome_size"]
unique_assembly_df_combined

In [ ]:
# from scipy.stats import mannwhitneyu
# from statannotations.Annotator import Annotator

# unique_assembly_df_combined["afr_group"] = unique_assembly_df_combined["population"].apply(
#     lambda x: "AFR" if x == "AFR" else "Rest"
# )
# method_order = list(unique_assembly_df_combined["method"].unique())
# hue_order = ["AFR", "Rest"]
# metrics = [
#     ("unique_motifs", "Unique G4s"),
#     # ("unique_motifs_per_mb", "Unique G4s / Mb"),
# ]
# fig, axes = plt.subplots(nrows=1, ncols=1, figsize=(11, 4.8))
# for ax, (metric, label) in zip(axes, metrics):
#     data = unique_assembly_df_combined.dropna(subset=[metric])
#     print(f"\n--- {label} ---")
#     for m in method_order:
#         sub = data[data["method"] == m]
#         afr = sub.loc[sub["afr_group"] == "AFR", metric].values
#         rest = sub.loc[sub["afr_group"] == "Rest", metric].values
#         u, p = mannwhitneyu(afr, rest, alternative="two-sided")
#         delta = 2 * u / (len(afr) * len(rest)) - 1
#         print(f"{m}: U={u:.1f}, p={p:.2e}, delta={delta:.3f} "
#               f"(n_AFR={len(afr)}, n_Rest={len(rest)})")
#     plot_params = dict(
#         data=data,
#         x="method",
#         y=metric,
#         hue="afr_group",
#         order=method_order,
#         hue_order=hue_order,
#     )
#     sns.boxplot(
#         **plot_params,
#         palette={"AFR": superpopulation_colors["AFR"], 
#                  "Rest": "dimgray"},
#         width=0.5,
#         linewidth=1.5,
#         flierprops=dict(marker="o", markersize=3, alpha=0.4),
#         ax=ax,
#     )

#     pairs = [((m, "AFR"), (m, "Rest")) for m in method_order]
#     annotator = Annotator(ax, pairs, **plot_params)
#     annotator.configure(
#         test="Mann-Whitney",
#         text_format="star",
#         loc="inside",
#         fontsize=16,
#         line_width=1.2,
#         verbose=0,
#     )
#     annotator.apply_and_annotate()

#     ax.set_xlabel("")
#     ax.set_ylabel(label, fontsize=22)
#     ax.tick_params(axis="both", labelsize=20)
#     ax.grid(lw=0.4, alpha=0.6, zorder=0)
#     ax.set_axisbelow(True)
#     ax.get_legend().remove()

# axes[0].legend(
#     *axes[0].get_legend_handles_labels(),
#     title="",
#     fontsize=15,
#     frameon=False,
#     loc="upper right",
# )
# sns.despine()
# plt.tight_layout()
# fig.savefig(
#     target_figures / "unique_assembly_g4_AFR_vs_Rest_raw_and_per_mb.pdf",
#     format="pdf",
#     transparent=True,
#     bbox_inches="tight",
#     dpi=600,
# )
# plt.show()

In [ ]:
unique_df_combined

In [ ]:
from scipy.stats import mannwhitneyu
from statannotations.Annotator import Annotator
from matplotlib.lines import Line2D

unique_df_combined["afr_group"] = unique_df_combined["population"].apply(
    lambda x: "AFR" if x == "AFR" else "Rest"
)

metric, label = "unique_motifs", "Unique G4s"
method_order = list(unique_df_combined["method"].unique())
hue_order = ["AFR", "Rest"]
palette = {"AFR": superpopulation_colors["AFR"], "Rest": "#9AA0A6"}

data = unique_df_combined.dropna(subset=[metric])

print(f"\n--- {label} ---")
for m in method_order:
    sub = data[data["method"] == m]
    afr = sub.loc[sub["afr_group"] == "AFR", metric].values
    rest = sub.loc[sub["afr_group"] == "Rest", metric].values
    u, p = mannwhitneyu(afr, rest, alternative="two-sided")
    delta = 2 * u / (len(afr) * len(rest)) - 1
    print(
        f"{m}: U={u:.1f}, p={p:.2e}, delta={delta:.3f} "
        f"(n_AFR={len(afr)}, n_Rest={len(rest)})"
    )

plot_params = dict(
    data=data,
    x="method",
    y=metric,
    hue="afr_group",
    order=method_order,
    hue_order=hue_order,
)

fig, ax = plt.subplots(figsize=(9, 5.4))

sns.boxplot(
    **plot_params,
    palette=palette,
    width=0.62,
    gap=0.18,
    linewidth=1.6,
    linecolor="#2B2B2B",
    fliersize=0,
    showcaps=False,
    whiskerprops=dict(linewidth=1.4, solid_capstyle="round"),
    medianprops=dict(linewidth=2.4, color="#1A1A1A", solid_capstyle="round"),
    boxprops=dict(alpha=0.65),
    ax=ax,
    zorder=2,
)

sns.stripplot(
    **plot_params,
    dodge=True,
    jitter=0.18,
    size=4.2,
    alpha=0.75,
    linewidth=0.5,
    edgecolor="white",
    palette={k: v for k, v in palette.items()},
    legend=False,
    ax=ax,
    zorder=3,
)

pairs = [((m, "AFR"), (m, "Rest")) for m in method_order]
annotator = Annotator(ax, pairs, **plot_params)
annotator.configure(
    test="Mann-Whitney",
    text_format="star",
    loc="inside",
    fontsize=17,
    line_width=1.3,
    line_height=0.015,
    text_offset=1.0,
    color="#2B2B2B",
    verbose=0,
)
annotator.apply_and_annotate()

ax.set_xlabel("")
ax.set_ylabel(label, fontsize=22, labelpad=10)
ax.tick_params(axis="both", labelsize=20, length=5, width=1.2)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax.margins(y=0.10)
ax.grid(axis="y", lw=0.4, alpha=0.6)
ax.grid(axis="x", visible=False)
ax.set_axisbelow(True)

n_counts = data.groupby(["method", "afr_group"]).size().unstack(fill_value=0)
ax.set_xticks(range(len(method_order)))
ax.set_xticklabels(
    [
        f"{m}\n$n$={n_counts.loc[m, 'AFR']} / {n_counts.loc[m, 'Rest']}"
        for m in method_order
    ],
    fontsize=20,
)

handles = [
    Line2D(
        [],
        [],
        marker="s",
        markersize=13,
        markerfacecolor=palette[g],
        markeredgecolor="#2B2B2B",
        markeredgewidth=1.2,
        linestyle="none",
        label=g,
    )
    for g in hue_order
]
ax.legend(
    handles=handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.13),
    ncol=2,
    fontsize=18,
    frameon=False,
    handletextpad=0.4,
    columnspacing=1.6,
)

sns.despine(ax=ax, trim=False)
for side in ("left", "bottom"):
    ax.spines[side].set_linewidth(1.2)

fig.savefig(
    target_figures / "unique_assembly_g4_AFR_vs_Rest.pdf",
    format="pdf",
    transparent=True,
    bbox_inches="tight",
    dpi=600,
)
plt.show()


## Jaccard Index between Haplotypes

In [ ]:
import itertools
from tqdm import tqdm
import pandas as pd

# Quadparser
samples = list(sequences_per_sample_quad.keys())
jaccard_indices = {}
for sample1, sample2 in itertools.combinations(samples, 2):
    inter = len(sequences_per_sample_quad[sample1] & sequences_per_sample_quad[sample2])
    union = len(sequences_per_sample_quad[sample1] | sequences_per_sample_quad[sample2])
    j = inter / union if union else 0.0
    jaccard_indices[(sample1, sample2)] = j
    jaccard_indices[(sample2, sample1)] = j
for s in samples:
    jaccard_indices[(s, s)] = 1.0

In [ ]:
from termcolor import colored 
MODE = "Quadparser"
jaccard_df = (
    pd.Series(jaccard_indices)
    .to_frame(name="jaccard")
    .reset_index()
    .rename(columns={"level_0": "sampleA",
                     "level_1": "sampleB"})
)
dest = f"{target_data}/jaccard_pairwise_haplotypes_unique_motifs_{MODE}.csv.gz"
jaccard_df.to_csv(dest,
                sep=",",
                compression="gzip",
                mode="w",
                index=False,
                header=True
            )
print(colored(f"Pairwise shared motifs data have been saved at `{dest}`.", "green"))

In [ ]:
# G4Hunter 
samples = list(sequences_per_sample.keys())
jaccard_indices = {}
for sample1, sample2 in itertools.combinations(samples, 2):
    inter = len(sequences_per_sample[sample1] & sequences_per_sample[sample2])
    union = len(sequences_per_sample[sample1] | sequences_per_sample[sample2])
    j = inter / union if union else 0.0
    jaccard_indices[(sample1, sample2)] = j
    jaccard_indices[(sample2, sample1)] = j
for s in samples:
    jaccard_indices[(s, s)] = 1.0

In [ ]:
from termcolor import colored 
MODE = "G4Hunter"
jaccard_df = (
    pd.Series(jaccard_indices)
    .to_frame(name="jaccard")
    .reset_index()
    .rename(columns={"level_0": "sampleA",
                     "level_1": "sampleB"})
)
dest = f"{target}/jaccard_pairwise_haplotypes_unique_motifs_{MODE}.csv.gz"
jaccard_df.to_csv(dest,
                sep=",",
                compression="gzip",
                mode="w",
                index=False,
                header=True
            )
print(colored(f"Pairwise shared motifs data have been saved at `{dest}`.", "green"))

In [ ]:
jaccard_df = pd.read_csv(f"{target_data}/jaccard_pairwise_haplotypes_unique_motifs_G4HUNTER.csv")
jaccard_df

In [ ]:
jaccard_quad_df = pd.read_csv(f"{target}/jaccard_pairwise_haplotypes_unique_motifs_Quadparser.csv.gz")
jaccard_quad_df

In [ ]:
# G4Hunter
jaccard_pivot_df = jaccard_df.pivot(
                            index="sampleA", 
                            columns="sampleB", 
                            values="jaccard")
row_colors = jaccard_pivot_df.index.map(lambda x: ancestries[x.split("#")[0]]).map(superpopulation_colors)
row_colors_x = jaccard_pivot_df.index.map(lambda x: x.split("#")[1]).map(haplotype_colors)
row_colors_y = jaccard_pivot_df.index.map(lambda x: sex_pop_meta[x.split("#")[0]]).map(sex_colors)

col_colors = jaccard_pivot_df.columns.map(lambda x: ancestries[x.split("#")[0]]).map(superpopulation_colors)
col_colors_x = jaccard_pivot_df.columns.map(lambda x: x.split("#")[1]).map(haplotype_colors)
col_colors_y = jaccard_pivot_df.columns.map(lambda x: sex_pop_meta[x.split("#")[0]]).map(sex_colors)
jaccard_pivot_df

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# row_colors = jaccard_pivot_df.index.map(lambda x: metadata[x.split(".")[0]]).map(superpopulation_colors)
# col_colors = jaccard_pivot_df.columns.map(lambda x: metadata[x.split(".")[0]]).map(superpopulation_colors)

g = sns.clustermap(data=jaccard_pivot_df, 
                   row_colors=[row_colors_y, row_colors_x, row_colors], 
                   col_colors=[col_colors_y, col_colors_x, col_colors], 
                   cmap="magma", 
                   # method="average",
                   robust=True,
                   cbar_kws={'label': 'Jaccard Index'}, 
                   figsize=(11, 11))
g.ax_row_dendrogram.set_visible(False)
g.ax_col_dendrogram.set_visible(False)

# Set the size of the ticks
g.ax_heatmap.tick_params(axis='x', labelsize=13, rotation=90)
g.ax_heatmap.tick_params(axis='y', labelsize=13, rotation=0)
g.ax_heatmap.set_xticks([])
g.ax_heatmap.set_yticks([])
g.ax_heatmap.set_ylabel("Haplotypes")
g.ax_heatmap.set_xlabel("Haplotypes")
g.ax_heatmap.xaxis.label.set_size(24)
g.ax_heatmap.yaxis.label.set_size(24)
g.ax_cbar.set_position([0.03, 0.2, 0.05, 0.5])
g.cax.tick_params(labelsize=18)
g.ax_cbar.yaxis.label.set_size(24)
plt.gcf().savefig(target_figures.joinpath(f"G4Hunter_clustermap_jaccard_g4.pdf"), 
                  transparent=True,
                  bbox_inches='tight', 
                  format="pdf", 
                  dpi=600)

In [ ]:
# Quadparser
jaccard_quad_df = pd.read_csv(f"{target_data}/jaccard_pairwise_haplotypes_unique_motifs_Quadparser.csv.gz")
jaccard_pivot_quad_df = jaccard_quad_df.pivot(
                            index="sampleA", 
                            columns="sampleB", 
                            values="jaccard")
row_colors = jaccard_pivot_quad_df.index.map(lambda x: ancestries[x.split("#")[0]]).map(superpopulation_colors)
row_colors_x = jaccard_pivot_quad_df.index.map(lambda x: x.split("#")[1]).map(haplotype_colors)
row_colors_y = jaccard_pivot_quad_df.index.map(lambda x: sex_pop_meta[x.split("#")[0]]).map(sex_colors)

col_colors = jaccard_pivot_quad_df.columns.map(lambda x: ancestries[x.split("#")[0]]).map(superpopulation_colors)
col_colors_x = jaccard_pivot_quad_df.columns.map(lambda x: x.split("#")[1]).map(haplotype_colors)
col_colors_y = jaccard_pivot_quad_df.columns.map(lambda x: sex_pop_meta[x.split("#")[0]]).map(sex_colors)
jaccard_pivot_quad_df

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# row_colors = jaccard_pivot_df.index.map(lambda x: metadata[x.split(".")[0]]).map(superpopulation_colors)
# col_colors = jaccard_pivot_df.columns.map(lambda x: metadata[x.split(".")[0]]).map(superpopulation_colors)
g = sns.clustermap(data=jaccard_pivot_quad_df, 
                   row_colors=[row_colors_y, row_colors_x, row_colors], 
                   col_colors=[col_colors_y, col_colors_x, col_colors], 
                   cmap="magma", 
                   # method="average",
                   robust=True,
                   cbar_kws={'label': 'Jaccard Index'}, 
                   figsize=(11, 11))
g.ax_row_dendrogram.set_visible(False)
g.ax_col_dendrogram.set_visible(False)

# Set the size of the ticks
g.ax_heatmap.tick_params(axis='x', labelsize=13, rotation=90)
g.ax_heatmap.tick_params(axis='y', labelsize=13, rotation=0)
g.ax_heatmap.set_xticks([])
g.ax_heatmap.set_yticks([])
g.ax_heatmap.set_ylabel("Haplotypes")
g.ax_heatmap.set_xlabel("Haplotypes")
g.ax_heatmap.xaxis.label.set_size(24)
g.ax_heatmap.yaxis.label.set_size(24)
g.ax_cbar.set_position([0.03, 0.2, 0.05, 0.5])
g.cax.tick_params(labelsize=18)
g.ax_cbar.yaxis.label.set_size(24)
plt.gcf().savefig(target_figures.joinpath("Quadparser_clustermap_jaccard_g4.pdf"), 
                  transparent=True,
                  bbox_inches='tight', 
                  format="pdf", 
                  dpi=600)

## Conservation

In [ ]:
afr_cols = [col for col in conservation_df.columns if col.split('#')[0] in ancestries and ancestries[col.split('#')[0]] == 'AFR']
non_afr_cols = [col for col in conservation_df.columns if col.split('#')[0] in ancestries and ancestries[col.split('#')[0]] != 'AFR']
amr_cols = [col for col in conservation_df.columns if col.split('#')[0] in ancestries and ancestries[col.split('#')[0]] == 'AMR']

# conservation_scored_df = conservation_df.with_columns([
#     (
#         pl.col(col).map_elements(lambda x: x.split(",")[1] if isinstance(x, str) else '.', return_dtype=str)
#     ).alias(col) 
#     for col in (afr_cols + non_afr_cols)
# ])
conservation_scored_df = conservation_df.with_columns([
    pl.col(col)
      .str.split(",")           # Split into a list: ["some_id", "score"]
      .list.get(1)
      .str.replace("%", "")
    .cast(pl.Float64)
      .alias(col)
    for col in (afr_cols + non_afr_cols)
])
conservation_scored_df

In [ ]:
conservation_df = conservation_df.with_columns(
                        pl.col("motif_hit_info")
                                .str.split(":").list.get(0).alias("seqID"),

                              pl.col("motif_hit_info")
                                    .str.split(":").list.get(1)
                                     .str.split(",").list.get(0)\
                                     .str.split("-").list.get(0).cast(pl.Int32).alias("start"),
    
                              pl.col("motif_hit_info").str.split(":").list.get(1)\
                                                      .str.split(",").list.get(0)\
                                                      .str.split("-").list.get(1).cast(pl.Int32).alias("end")
                            
                            )\
                            .with_columns(
                                    pl.col("motif_name").str.len_chars().alias("sequence_length")
                            )                          
total_haplotypes = len([col for col in conservation_df.columns if col.split('#')[0] in ancestries])

if not isinstance(sex_pop, dict):
    sex_pop = sex_pop.to_dict()
    
conservation_df = conservation_df.with_columns(
                                pl.sum_horizontal([pl.col(col).is_not_null().cast(pl.Int8) for col in conservation_df.columns if col.split('#')[0] in ancestries]).alias("non_null_count"),
                                pl.sum_horizontal([int(sex_pop[col.split('#')[0]] == "male") for col in conservation_df.columns if col.split("#")[0] in sex_pop]).alias("total_from_male"),
                                pl.sum_horizontal([1 for col in conservation_df.columns if col.split('#')[0] in ancestries]).alias("total"),
                                total_AFR=pl.sum_horizontal([pl.col(col).is_not_null().cast(pl.Int8) for col in conservation_df.columns if (col.split('#')[0] in ancestries and ancestries[col.split('#')[0]] == 'AFR')])
                            )\
                                .with_columns(
                                            (pl.col("total_from_male") // 2).alias("total_haplotypes_with_Y"),
                                )\
                                .with_columns(
                                            (88 - pl.col("total_haplotypes_with_Y")).alias("total_haplotypes_with_X"),
                                            pl.lit(88).alias("total_haplotypes_autosomal")
                                )\
                                .with_columns(
                                            pl.when(pl.col("seqID") == "chrX")
                                              .then(
                                                      (pl.col("non_null_count") * 1e2 / pl.col("total_haplotypes_with_X")),
                                              )
                                            .otherwise(
                                                    pl.when(pl.col("seqID") == "chrY")
                                                .then(
                                                        (pl.col("non_null_count") * 1e2 / pl.col("total_haplotypes_with_Y"))
                                                )
                                                .otherwise(
                                                        (pl.col("non_null_count") * 1e2 / pl.col("total_haplotypes_autosomal"))
                                                )
                                            ).alias("haplotype_presence"),
                                )\
                                .sort(["seqID", "start"], descending=False)

conservation_df

In [ ]:
from Bio.Seq import Seq 

conservation_df = conservation_df.with_columns(
    canonical=pl.col("motif_name").map_elements(lambda seq: seq if seq.count("G") >= seq.count("C") else str(Seq(seq).reverse_complement()),
    return_dtype=pl.Utf8)
)
conservation_df

In [ ]:
afr_cols = [col for col in conservation_df.columns if col.split('#')[0] in ancestries and ancestries[col.split('#')[0]] == 'AFR']
non_afr_cols = [col for col in conservation_df.columns if col.split('#')[0] in ancestries and ancestries[col.split('#')[0]] != 'AFR']
amr_cols = [col for col in conservation_df.columns if col.split('#')[0] in ancestries and ancestries[col.split('#')[0]] == 'AMR']

conservation_scored_df = conservation_df.with_columns([
    pl.col(col).str.split(",").list.get(1, null_on_oob=True).fill_null(".")
    for col in (afr_cols + non_afr_cols + ["GRCh38"])
])

In [ ]:
conservation_scoredd_df = conservation_scored_df.with_columns([
    pl.col(col)
      .str.strip_suffix("%")
      .cast(pl.Float64, strict=False)
      .alias(col)
    for col in (afr_cols + non_afr_cols + ["GRCh38"])
])
conservation_scoredd_df

In [ ]:
valid_haplotypes = [col for col in conservation_df.columns if col.split('#')[0] in ancestries]
len(valid_haplotypes)

In [ ]:
conservation_scoredd_df = conservation_scoredd_df.with_columns(
    AFR_avg_cons = pl.mean_horizontal(afr_cols),
    NonAFR_avg_cons = pl.mean_horizontal(non_afr_cols),
    avg_cons=pl.mean_horizontal(valid_haplotypes),
    avgs_cons_all=pl.mean_horizontal([col for col in valid_haplotypes] + ["GRCh38"]),
    
    # Keep track of how many haplotypes actually had the G4
    AFR_valid_count = pl.sum_horizontal([pl.col(col).is_not_null().cast(pl.Int32) for col in afr_cols]),
    NonAFR_valid_count = pl.sum_horizontal([pl.col(col).is_not_null().cast(pl.Int32) for col in non_afr_cols])
)
conservation_scoredd_df = conservation_scoredd_df.with_columns(
    pl.sum_horizontal([
        (pl.col(col) == 100.0).cast(pl.Int32) for col in afr_cols
    ]).alias("AFR_100pct_count"),
    pl.sum_horizontal([
        (pl.col(col) == 100.0).cast(pl.Int32) for col in amr_cols
    ]).alias("AMR_100pct_count"),
    pl.sum_horizontal([
        (pl.col(col) < 60.0).cast(pl.Int32) for col in afr_cols
    ]).alias("AFR_60pct_count"),
)
conservation_scoredd_df

In [ ]:
conservation_scoredd_df = conservation_scoredd_df.with_columns(
    AFR_avg_cons = pl.mean_horizontal(afr_cols),
    NonAFR_avg_cons = pl.mean_horizontal(non_afr_cols),
    avg_cons=pl.mean_horizontal(valid_haplotypes),
    avgs_cons_all=pl.mean_horizontal([col for col in valid_haplotypes] + ["GRCh38"]),
    
    # Keep track of how many haplotypes actually had the G4
    AFR_valid_count = pl.sum_horizontal([pl.col(col).is_not_null().cast(pl.Int32) for col in afr_cols]),
    NonAFR_valid_count = pl.sum_horizontal([pl.col(col).is_not_null().cast(pl.Int32) for col in non_afr_cols]),
    AMR_valid_count=pl.sum_horizontal([pl.col(col).is_not_null().cast(pl.Int32) for col in amr_cols]),
)
conservation_scoredd_df = conservation_scoredd_df.with_columns(
    AFR_avg_cons = pl.mean_horizontal(afr_cols),
    NonAFR_avg_cons = pl.mean_horizontal(non_afr_cols),
    AMR_avg_cons=pl.mean_horizontal(amr_cols),
    avg_cons=pl.mean_horizontal(valid_haplotypes),
    avgs_cons_all=pl.mean_horizontal([col for col in valid_haplotypes] + ["GRCh38"]),
    
    # Keep track of how many haplotypes actually had the G4
    AFR_valid_count = pl.sum_horizontal([pl.col(col).is_not_null().cast(pl.Int32) for col in afr_cols]),
    NonAFR_valid_count = pl.sum_horizontal([pl.col(col).is_not_null().cast(pl.Int32) for col in non_afr_cols]),
    AMR_valid_count=pl.sum_horizontal([pl.col(col).is_not_null().cast(pl.Int32) for col in amr_cols]),
)
conservation_scoredd_df = conservation_scoredd_df.with_columns(
    pl.sum_horizontal([
        (pl.col(col) == 100.0).cast(pl.Int32) for col in afr_cols
    ]).alias("AFR_100pct_count"),
    pl.sum_horizontal([
        (pl.col(col) == 100.0).cast(pl.Int32) for col in amr_cols
    ]).alias("AMR_100pct_count"),
    pl.sum_horizontal([
        (pl.col(col) < 80.0).cast(pl.Int32) for col in afr_cols
    ]).alias("AFR_80pct_count"),
    pl.sum_horizontal([
        (pl.col(col) < 90.0).cast(pl.Int32) for col in amr_cols
    ]).alias("AMR_80pct_count"),
)

conservation_scoredd_df

In [ ]:
from pathlib import Path 
from pybedtools import BedTool
import pybedtools

dataset_path = Path(os.getenv("SCRATCH")) / "data"
G4HUNTER = dataset_path / "pG4s_extractions" / "g4hunter" / "chm13v2_g4hunter.txt.gz"
g4_df = pd.read_table(G4HUNTER)
g4_bed = BedTool.from_dataframe(g4_df[["seqID", "start", "end"]]).sort()

In [ ]:
conservation_df_ref_g4 = conservation_scoredd_df\
                                    .with_columns(end=pl.col("end")+1)\
                                    .join(pl.from_pandas(g4_df),
                                            left_on=["seqID", "start", "end"],
                                            right_on=["seqID", "start", "end"],
                                            how="inner"
                                    )
conservation_df_ref_g4.shape

In [ ]:
g4_bed = BedTool.from_dataframe(g4_df.drop(columns=['NBR'])).sort() 

In [ ]:
AFR_haplotypes = [col for col in valid_haplotypes if ancestries[col.split('#')[0]] == 'AFR']
total_AFR_global = len(AFR_haplotypes)
total_non_AFR_global = len(valid_haplotypes) - total_AFR_global
total_non_AFR_global

In [ ]:
from pybedtools import BedTool

conservation_bed = BedTool.from_dataframe(conservation_df.with_columns(end=pl.col("end")+1)\
                                                      .select(["seqID", "start", "end", "motif_name", "score", "haplotype_presence", "non_null_count"]).to_pandas()
                                         ).sort()

g4_bed = BedTool.from_dataframe(g4_df.drop(columns=['NBR'])).sort()
g4_conserved_with_ref_bed = conservation_bed.intersect(g4_bed, f=1.0, u=True)
g4_conserved_df = pd.read_table(g4_conserved_with_ref_bed.fn,
                                header=None,
                                names=["seqID", "start", "end", "motif_name", "align_score", "haplotype_presence", "non_null_count"]
                               ).query("seqID != 'chrY'")
g4_conserved_df.shape

In [ ]:
# Analysis 1: Truly AMR-specific
amr_specific = conservation_scoredd_df.filter(
    pl.col("AMR_valid_count") >= 6,
    pl.col("AFR_valid_count") == 0,
)
amr_specific

In [ ]:
regions_df = pd.read_table(Path(os.getenv('WORK')).joinpath("compartments_coords.tsv.gz"))
regions_bed = BedTool.from_dataframe(regions_df).sort()
regions_df

In [ ]:
from tqdm import tqdm 
FASTA = dataset_path / "fasta" / "hs1.fa.gz"
FASTA = dataset_path / "hs1.fa.gz"
assert FASTA.is_file()
# # # #
seq_sizes = pd.read_table(dataset_path / "genome.txt", 
                          header=None)
seq_sizes = dict(zip(seq_sizes[0], seq_sizes[1]))
with open("genome.txt", "w") as f:
    for seqID, length in seq_sizes.items():
        f.write(f"{seqID}\t{length}\n")

In [ ]:
MIN_SCORE = 80.0 
MIN_HAPLOTYPE_PRESENCE = 10.0
g4_not_conserved_df = g4_conserved_df[(g4_conserved_df["haplotype_presence"] <= MIN_HAPLOTYPE_PRESENCE)]
g4_not_conserved_df

In [ ]:
not_aligned = pd.read_table(
                    g4_bed.intersect(conservation_bed, v=True).fn,
                    header=None,
                    names=["seqID", "start", "end", "sequence", "length", "score"]
).query("seqID != 'chrY'") 
not_aligned

In [ ]:
not_conserved_g4 = pd.concat([
                    not_aligned[["seqID", "start", "end", "sequence"]],

                    
])
not_conserved_g4["sequence"] = not_conserved_g4["sequence"].str.upper()

In [ ]:
BedTool.from_dataframe(not_conserved_g4).sort().merge().count()

In [ ]:
conservation_df_ref_g4 = (
                    conservation_df_ref_g4
                        .with_columns(
                            cons_enrichment=pl.col("AFR_avg_cons") / pl.col("NonAFR_avg_cons")
                        )
                        .with_columns(AFR_presence=pl.col("total_AFR") / total_AFR_global,
                                     non_AFR_presence=(pl.col("non_null_count") - pl.col("total_AFR"))/ total_non_AFR_global
                                     )
                        .with_columns(
                            presence_enrichment=pl.col("AFR_presence") / pl.col("non_AFR_presence")
                        )
)
conservation_df_ref_g4                             

## Conservation per Region

In [ ]:
regions_df = pd.read_table(Path(os.getenv('WORK')).joinpath("compartments_coords.tsv.gz"))
regions_bed = BedTool.from_dataframe(regions_df).sort()
regions_df

In [ ]:
from tqdm import tqdm 
from Bio.SeqIO.FastaIO import SimpleFastaParser

def _open_maybe_gzip(file_path):
    if str(file_path).endswith(".gz"):
        import gzip
        return gzip.open(file_path, "rt")
    else:
        return open(file_path, "r")

def parse_fasta(fasta_path):
    with _open_maybe_gzip(fasta_path) as handle:
        for title, seq in SimpleFastaParser(handle):
            yield title.split()[0], seq.lower()
            
indir = Path(f"{os.getenv('SCRATCH')}/g4_t2t_revisions_data")

FASTA_hs1 = Path(f"{indir}/fasta/hs1.fa.gz")
assert FASTA_hs1.is_file()
# # # #
seq_sizes_hs1 = dict()
chrom_sequences_hs1 = dict()
genome_size_hs1 = 0


for seqID, seq in tqdm(parse_fasta(FASTA_hs1), total=24):
    # chrom_sequences_hs1[seqID] = seq.upper()
    seq_sizes_hs1[seqID] = len(seq)
    genome_size_hs1 += seq_sizes_hs1[seqID]
genome_size_hs1

In [ ]:
import pyranges as pr
def calculate_gw_density(df, genome_size):
    df_pr = pr.from_dict({
        "Chromosome": df["seqID"],
        "Start": df["start"],
        "End": df["end"]
    })
    df_pr_merged = df_pr.merge()
    total_bases = (df_pr_merged.End - df_pr_merged.Start).sum()
    gw_density = total_bases * 1e3 / genome_size    
    return gw_density

g4_gw_density = calculate_gw_density(g4_df, genome_size_hs1)
g4_gw_density

In [ ]:
import math 
# Coverage of all G4s per compartment (full denominator)
COVERAGE_FIELDS = ["totalHits", "overlappingBp", "compartmentLength", "coverage"]
not_conserved_bed = BedTool.from_dataframe(
    not_conserved_g4[["seqID", "start", "end"]].drop_duplicates()
).sort()

coverage_df = pd.read_table(
    regions_bed.coverage(not_conserved_bed).fn,
    header=None,
    names=["seqID", "start", "end", "comp"] + COVERAGE_FIELDS
).query("seqID != 'chrY'").groupby("comp", as_index=False).agg({
    "overlappingBp": "sum",
    "compartmentLength": "sum",
    "totalHits": "sum"
})

cov_all_g4 = (
    pd.read_table(
        regions_bed.coverage(g4_bed).fn,
        header=None,
        names=["seqID", "start", "end", "comp"] + COVERAGE_FIELDS
        )
    .query("seqID != 'chrY'")
    .groupby("comp", as_index=False)
    .agg({
                "overlappingBp": "sum",
                "compartmentLength": "sum",
                "totalHits": "sum"
    })
    .rename(columns={
            "overlappingBp": "overlappingBp_global",
            "compartmentLength": "compartmentLength_global",
            "totalHits": "totalHits_global"
    })
)

coverage_df = coverage_df.merge(cov_all_g4, on="comp", how="left")
coverage_df = coverage_df.query("totalHits_global > 20").reset_index(drop=True)

coverage_df["not_conserved_perc"]    = 1e2 * coverage_df["totalHits"] / coverage_df["totalHits_global"]
coverage_df["density_not_conserved"] = 1e6 * coverage_df["overlappingBp"] / coverage_df["compartmentLength"]
coverage_df["density_all_g4"]        = 1e6 * coverage_df["overlappingBp_global"] / coverage_df["compartmentLength_global"]


In [ ]:
# Fold enrichment: not-conserved density vs all-G4 density, per compartment
import math 
coverage_df["fold_enrichment"] = coverage_df["density_all_g4"] / (1e3 * g4_gw_density)
coverage_df["log10_fe"]        = coverage_df["fold_enrichment"].apply(lambda x: math.log10(x) if x > 0 else float("nan"))
coverage_df = coverage_df.sort_values(by=["fold_enrichment"], ascending=False).reset_index(drop=True)
coverage_df

In [ ]:
%matplotlib inline 
import matplotlib.pyplot as plt 
cmap = plt.cm.PuRd
data = coverage_df.copy()
norm = plt.Normalize(data["not_conserved_perc"].min(), data["not_conserved_perc"].max())
colors = cmap(norm(data["not_conserved_perc"]))

fig, ax = plt.subplots(figsize=(9, 8))
bars = ax.barh(data["comp"], data["log10_fe"],
               color=colors, edgecolor="black", linewidth=2, height=0.9)

for i, bar in enumerate(bars):
    width = bar.get_width()
    offset = -0.22 if width < 0 else 0.22
    ax.text(width + offset, bar.get_y() + bar.get_height() / 12,
            f"{data['not_conserved_perc'].iloc[i]:.1f}%",
            ha="center", va="bottom", fontsize=13, color="black")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(
    data["not_conserved_perc"].min(), data["not_conserved_perc"].max()
))
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label("Not aligned (%)", fontsize=20)
cbar.ax.tick_params(labelsize=16)

ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.grid(lw=0.4, alpha=0.6, zorder=0)
ax.set_axisbelow(True)
ax.set_xlabel(r"$\log_{10}(\mathrm{Fold\ Enrichment})$", fontsize=17)
ax.tick_params(axis="y", labelsize=13)
ax.tick_params(axis="x", labelsize=16)
ax.set_ylim(ax.get_ylim()[0] + 0.7, ax.get_ylim()[1] - 0.9)
ax.set_xlim(ax.get_xlim()[0] - 0.5, ax.get_xlim()[1] + 0.5)
plt.tight_layout()
plt.show()
fig.savefig(target_figures / "compartment_enrichment_not_conserved_g4.pdf", 
            format="pdf", 
            dpi=600, 
            transparent=True, 
            bbox_inches="tight")